# 🫀 Heart Attack Risk Predictor — RAG Demo

This notebook demonstrates the **Retrieval-Augmented Generation (RAG)** system integrated with your sklearn heart attack risk predictor.

## What is RAG?
RAG = **Retrieval-Augmented Generation**
- A technique that combines a **vector database** (your medical knowledge) with an **LLM** (AI language model)
- Instead of trusting the AI's general knowledge, RAG **retrieves relevant facts** from your curated medical documents first, then generates a grounded answer.

## Architecture
```
User Question
     ↓
[Embedding Model] → Searches ChromaDB (Vector Store)
                          ↓
               [Top 4 Relevant Chunks]
                          ↓
       [LLM Prompt = Medical Context + Question]
                          ↓
             [LLM Answer — Grounded in Facts]
```

## Files Needed
- `heart.csv` — dataset  
- `heart.pkl` — saved sklearn model  
- `rag_knowledge_base.md` — medical knowledge document  
- `rag_pipeline.py` — RAG engine (core code)  

---
## Step 1: Install Dependencies
> **Run this cell only once.** Restart the kernel after installation.

In [1]:
# Install all RAG dependencies
# This may take 2-3 minutes on first run
!pip install langchain langchain-community langchain-google-genai
!pip install chromadb google-generativeai
!pip install tiktoken pypdf python-dotenv

print("\n✅ All dependencies installed! Restart kernel now.")

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_classic-1.0.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached google_genai-1.72.0-py3-none-any.whl.metadata (52 kB)
  Using cached google_auth-2.49.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached websockets-16.0-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
Using cached langc

In [4]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers"])



0

In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-groq"])


0

---
## Step 2: Configure Your API Key

### Get a FREE Google Gemini API key:
1. Go to: https://aistudio.google.com/app/apikey
2. Sign in with Google
3. Click **Create API Key**
4. Paste in the cell below

In [2]:
# Option A: Enter your API key directly (quick testing)
GROQ_API_KEY = "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"  # <-- Replace this!

# Option B: Load from .env file (more secure)
# from dotenv import load_dotenv
# import os
# load_dotenv('.env')
# GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

# Option A: Enter your API key directly (quick testing)
GROQ_API_KEY = "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"  # <-- Replace this!

# Option B: Load from .env file (more secure)
# from dotenv import load_dotenv
# import os
# load_dotenv('.env')
# GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

print(f"API Key set: {'✅' if GROQ_API_KEY and GROQ_API_KEY.startswith('gsk_') else '❌ Not set yet'}")

API Key set: ✅


---
## Step 3: Initialize the RAG Pipeline

This step:
1. Loads `rag_knowledge_base.md` and `heart.csv`
2. Splits docs into overlapping chunks (800 chars each)
3. Converts chunks to **embeddings** (numerical vectors) using Gemini
4. Stores them in **ChromaDB** (local database)

> First run takes ~20-30 seconds. Subsequent runs are instant (loads from cache).

In [13]:
from rag_pipeline import create_rag_pipeline

# Initialize the RAG system
rag = create_rag_pipeline(
    api_key=GROQ_API_KEY,
    provider="groq",     # or "openai" or "ollama"
    rebuild=False,          # Set True to force rebuild vector DB
    base_dir="."           # Current directory
)

print("\n🚀 RAG system ready!")

[RAG] Initializing provider: groq
[RAG] Loading local embedding model (all-MiniLM-L6-v2) — first run ~30s...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[RAG] ✓ Local embeddings ready
[RAG] ✓ LLM ready (groq)
[RAG] Loading existing vector store from .\chroma_db
[RAG] ✓ Loaded 84 chunks from cache
[RAG] ✓ QA chain ready. You can now call .query()

🚀 RAG system ready!


In [14]:
import pickle, pandas as pd, os

# Manually load the ML model into the rag object
if rag.model is None:
    model_path = os.path.join(".", "heart.pkl")
    data_path  = os.path.join(".", "heart.csv")
    
    if os.path.exists(model_path):
        with open(model_path, "rb") as f:
            rag.model = pickle.load(f)
        print(f"✅ ML model loaded: {type(rag.model).__name__}")
    else:
        print(f"❌ heart.pkl not found at: {os.path.abspath(model_path)}")

    if os.path.exists(data_path):
        rag.df = pd.read_csv(data_path)
        rag.df = rag.df.drop(["oldpeak", "slp", "thall"], axis=1, errors="ignore")
        print(f"✅ Dataset loaded: {len(rag.df)} patients")
    
    print(f"\nCurrent working directory: {os.getcwd()}")


✅ ML model loaded: LogisticRegression
✅ Dataset loaded: 303 patients

Current working directory: c:\Users\SATYAJIT NAYAK\.gemini\antigravity\scratch\antigravity_project


---
## Step 4: Ask General Medical Questions

Test the RAG system with different types of questions.

In [21]:
# ── Question 1: Feature explanation ──
answer = rag.query("What does chest pain type (cp) indicate about heart attack risk?")


Question: What does chest pain type (cp) indicate about heart attack risk?

Answer:
Based on the provided context, chest pain type (cp) is a categorical feature with values ranging from 0 to 3. The risk insight associated with cp is that higher values indicate a higher heart attack risk.

Specifically, the correlation with output is +0.434, which suggests that cp is a strong positive predictor of heart attack risk. This means that as the value of cp increases, the likelihood of a heart attack also increases.

Here's a breakdown of the cp values and their corresponding risk implications:

- **cp=0** (Typical angina): Classic presentation, but not necessarily the highest risk.
- **cp=1** (Atypical angina): May indicate a higher risk than typical angina, but still relatively lower than asymptomatic patients.
- **cp=2** (Non-anginal pain): Not related to coronary artery disease, so likely lower risk.
- **cp=3** (Asymptomatic): May have severe underlying disease (silent ischemia), indicati

In [22]:
# ── Question 2: Most important features ──
answer = rag.query("Which features are the strongest predictors of heart attack in this dataset?")


Question: Which features are the strongest predictors of heart attack in this dataset?

Answer:
Based on the provided medical context, I don't have explicit information about the strength of each feature as a predictor. However, I can infer that the 10 key clinical features used in the model are significant predictors of heart attack risk.

To provide a more accurate answer, I would need to refer to a separate section of the medical knowledge base that outlines the feature importance or correlation coefficients. Unfortunately, such information is not provided in the given context.

However, I can suggest that the following features are commonly associated with heart attack risk:

1. **Age**: Older age is a well-established risk factor for heart attacks.
2. **Diabetes**: Presence of diabetes is a significant predictor of heart attack risk.
3. **Hypertension**: High blood pressure is a major risk factor for heart attacks.
4. **Smoking**: Smoking is a well-known risk factor for heart att

In [23]:
# ── Question 3: Specific medical value ──
answer = rag.query("A patient has cholesterol of 300 mg/dL. Is this concerning for heart attack risk?")


Question: A patient has cholesterol of 300 mg/dL. Is this concerning for heart attack risk?

Answer:
**Disclaimer:** This response is for educational purposes only and not intended as medical advice. Please consult a qualified healthcare professional for personalized guidance.

Based on the provided context, a cholesterol level of 300 mg/dL falls into the "high" category, as it is ≥ 240 mg/dL. According to the risk insight, high LDL cholesterol contributes to plaque buildup in arteries (atherosclerosis), directly increasing heart attack risk.

Given this information, a cholesterol level of 300 mg/dL is concerning for heart attack risk, as it indicates a high level of LDL cholesterol that may contribute to plaque buildup in the arteries.

**Risk Assessment:** Based on the provided context, I would categorize this patient's heart attack risk as "increased" due to their high cholesterol level. However, please note that this is a single-factor assessment and does not take into account oth

In [24]:
# ── Question 4: Prevention ──
answer = rag.query("What lifestyle changes can reduce heart attack risk?")


Question: What lifestyle changes can reduce heart attack risk?

Answer:
**Disclaimer:** This information is for educational purposes only and should not be considered as medical advice. Consult a healthcare professional for personalized guidance.

Based on the provided context, several lifestyle changes can reduce heart attack risk:

1. **Regular exercise**: Engaging in physical activity can reduce heart attack risk by 35%. This is a significant benefit, making regular exercise a crucial aspect of heart health.
2. **Healthy diet**: Adopting a balanced diet that is low in saturated fat, sodium, and processed foods can help mitigate heart attack risk. Aiming for a diet rich in fruits, vegetables, whole grains, and lean protein sources can be beneficial.
3. **Weight management**: Maintaining a healthy weight through a combination of diet and exercise can help reduce cardiac workload and inflammation, both of which are associated with an increased risk of heart attack.
4. **Stress managem

In [25]:
# ── Question 5: ECG interpretation ──
answer = rag.query("My patient has restecg=1 (ST-T abnormality). What does this mean?")


Question: My patient has restecg=1 (ST-T abnormality). What does this mean?

Answer:
**Disclaimer:** This response is for educational purposes only and not intended as medical advice. Please consult a qualified healthcare professional for personalized guidance.

Based on the provided context, a restecg value of 1 indicates an ST-T wave abnormality. This suggests ischemia or injury to the heart. Specifically, it means that there are T wave inversions and/or ST elevation/depression greater than 0.05 mV on the patient's resting ECG.

**Risk Insight:** As mentioned in the context, ST-T abnormalities are a direct sign of cardiac ischemia, which increases the risk of heart attack.

**Risk Assessment:** Given the presence of an ST-T wave abnormality (restecg=1), the patient's risk of heart attack is moderately increased. This is based on the moderate positive correlation (+0.137) between abnormal ECG results and increased risk.

**Recommendation:** Further evaluation and monitoring are recom

---
## Step 5: Patient Risk Assessment

Input a patient's clinical values → get ML prediction + AI explanation.

In [19]:
# ── High-Risk Patient Example ──
high_risk_patient = {
    "age": 62,
    "sex": 1,           # Male
    "cp": 3,            # Asymptomatic (paradoxically high risk)
    "trtbps": 150,      # High blood pressure (hypertension)
    "chol": 285,        # High cholesterol
    "fbs": 1,           # Fasting blood sugar > 120 (diabetic)
    "restecg": 1,       # ST-T wave abnormality
    "thalachh": 165,    # Max heart rate
    "exng": 0,          # No exercise angina
    "caa": 2            # 2 major vessels blocked
}

result = rag.assess_patient(high_risk_patient, explain=True)
print(f"\nFinal Risk Level: {result.get('risk_level', 'N/A')}")


────────────────────────────────────────
ML Model Prediction: LOW RISK
Probability - High Risk: 0.5%
Probability - Low Risk:  99.5%
────────────────────────────────────────

Question: 
Based on this patient's clinical data, explain their heart attack risk assessment:

Patient Profile:
- Age: 62 years
- Sex: Male
- Chest Pain Type: asymptomatic (cp=3)
- Resting Blood Pressure: 150 mm Hg
- Cholesterol: 285 mg/dL
- Fasting Blood Sugar > 120: Yes
- Resting ECG: ST-T wave abnormality
- Max Heart Rate Achieved: 165 bpm
- Exercise Induced Angina: No
- Major Vessels Blocked (caa): 2
- Model Prediction: LOW RISK

What do these specific values indicate about their cardiovascular health?
Which features are most concerning and why?


Answer:
**Disclaimer:** This is for educational purposes only and not intended as medical advice. Please consult a healthcare professional for personalized guidance.

**Heart Attack Risk Assessment:**

Based on the provided patient data, we can analyze the risk facto

In [20]:
# ── Lower-Risk Patient Example ──
low_risk_patient = {
    "age": 35,
    "sex": 0,           # Female
    "cp": 1,            # Atypical angina
    "trtbps": 118,      # Normal blood pressure
    "chol": 195,        # Normal cholesterol
    "fbs": 0,           # Normal fasting blood sugar
    "restecg": 0,       # Normal ECG
    "thalachh": 175,    # Good max heart rate
    "exng": 0,          # No exercise angina
    "caa": 0            # No blocked vessels
}

result = rag.assess_patient(low_risk_patient, explain=True)


────────────────────────────────────────
ML Model Prediction: HIGH RISK
Probability - High Risk: 100.0%
Probability - Low Risk:  0.0%
────────────────────────────────────────

Question: 
Based on this patient's clinical data, explain their heart attack risk assessment:

Patient Profile:
- Age: 35 years
- Sex: Female
- Chest Pain Type: atypical angina (cp=1)
- Resting Blood Pressure: 118 mm Hg
- Cholesterol: 195 mg/dL
- Fasting Blood Sugar > 120: No
- Resting ECG: normal
- Max Heart Rate Achieved: 175 bpm
- Exercise Induced Angina: No
- Major Vessels Blocked (caa): 0
- Model Prediction: HIGH RISK

What do these specific values indicate about their cardiovascular health?
Which features are most concerning and why?


Answer:
**Disclaimer:** This is for educational purposes only, not medical advice. Please consult a healthcare professional for personalized guidance.

**Risk Assessment:**

Based on the provided clinical data, the patient's heart attack risk assessment is categorized as HIG

In [26]:
# ── Enter YOUR OWN Patient Data ──
my_patient = {
    "age": 50,        # Patient age
    "sex": 1,         # 0=Female, 1=Male
    "cp": 2,          # 0=typical angina, 1=atypical, 2=non-anginal, 3=asymptomatic
    "trtbps": 130,    # Resting blood pressure in mm Hg
    "chol": 230,      # Cholesterol in mg/dL
    "fbs": 0,         # Fasting blood sugar: 0=no, 1=yes (>120 mg/dL)
    "restecg": 0,     # 0=normal, 1=ST-T abnormality, 2=LVH
    "thalachh": 155,  # Maximum heart rate achieved
    "exng": 1,        # Exercise induced angina: 0=no, 1=yes
    "caa": 1          # Major vessels blocked: 0, 1, 2, or 3
}

result = rag.assess_patient(my_patient, explain=True)


────────────────────────────────────────
ML Model Prediction: HIGH RISK
Probability - High Risk: 100.0%
Probability - Low Risk:  0.0%
────────────────────────────────────────

Question: 
Based on this patient's clinical data, explain their heart attack risk assessment:

Patient Profile:
- Age: 50 years
- Sex: Male
- Chest Pain Type: non-anginal pain (cp=2)
- Resting Blood Pressure: 130 mm Hg
- Cholesterol: 230 mg/dL
- Fasting Blood Sugar > 120: No
- Resting ECG: normal
- Max Heart Rate Achieved: 155 bpm
- Exercise Induced Angina: Yes
- Major Vessels Blocked (caa): 1
- Model Prediction: HIGH RISK

What do these specific values indicate about their cardiovascular health?
Which features are most concerning and why?


Answer:
**Disclaimer:** This is for educational purposes only and not intended as medical advice. Please consult a qualified healthcare professional for personalized guidance.

**Heart Attack Risk Assessment:**

Based on the provided patient data, the risk assessment indicat

---
## Step 6: Debug — View Retrieved Chunks

See exactly which parts of the knowledge base the RAG system retrieves for a query.

In [27]:
# Direct similarity search (no LLM, just retrieval)
chunks = rag.search_knowledge_base(
    query="What is exercise induced angina?",
    k=3
)


--- Top 3 Retrieved Chunks for: 'What is exercise induced angina?' ---

[1] Similarity Score: 0.5409
Source: .\rag_knowledge_base.md
### 9. Exercise Induced Angina (`exng`)
- **Type**: Binary (1 = yes, 0 = no)
- **Medical meaning**: Whether the patient experienced chest pain/angina during exercise (e.g., treadmill stress test).
- **Risk insight**: Exercise-induced angina is a strong indicator of significant coronary artery disease. It means blood supply cannot meet demand during exertion.
- **Correlation with output**: -0.437. 

[2] Similarity Score: 0.5409
Source: .\rag_knowledge_base.md
### 9. Exercise Induced Angina (`exng`)
- **Type**: Binary (1 = yes, 0 = no)
- **Medical meaning**: Whether the patient experienced chest pain/angina during exercise (e.g., treadmill stress test).
- **Risk insight**: Exercise-induced angina is a strong indicator of significant coronary artery disease. It means blood supply cannot meet demand during exertion.
- **Correlation with output**: -0.437. 

[

In [29]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit"])


0

In [30]:
import subprocess, sys, os

# Install streamlit
subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit", "-q"])

# Launch app (runs in background)
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py"],
    cwd=os.getcwd()
)
print("✅ App launched! Open: http://localhost:8501")



✅ App launched! Open: http://localhost:8501


In [33]:
import subprocess, sys, os, time

os.chdir(r"C:\Users\SATYAJIT NAYAK\.gemini\antigravity\scratch\antigravity_project")

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)

poll = proc.poll()
print(f"Process status: {'Running ✅' if poll is None else f'Crashed ❌ (exit code {poll})'}")

# Read any output
try:
    out, err = proc.communicate(timeout=2)
    print("Output:", out.decode()[:500])
    print("Error:", err.decode()[:500])
except:
    print("(Process still running — checking output...)")


Process status: Running ✅
(Process still running — checking output...)


In [34]:
print(GROQ_API_KEY)   # Run this cell to see your key



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


---
## Step 7: Interactive Q&A Loop

Ask unlimited questions in a loop until you type 'quit'.

In [28]:
print("🫀 Heart Attack Risk Q&A (type 'quit' to stop)")
print("=" * 55)

while True:
    question = input("\nYour question: ").strip()
    if question.lower() in ["quit", "exit", "q"]:
        print("Goodbye! Stay heart-healthy! 💚")
        break
    if not question:
        continue


    
    rag.query(question)

🫀 Heart Attack Risk Q&A (type 'quit' to stop)
Goodbye! Stay heart-healthy! 💚


---
## Summary

### What you built:
| Component | Technology | Purpose |
|-----------|-----------|--------|
| ML Model | scikit-learn classifiers | Predict heart attack risk from patient data |
| Embeddings | Google Gemini `text-embedding-004` | Convert text to numerical vectors |
| Vector Store | ChromaDB (local) | Store and search embedded knowledge chunks |
| LLM | Google Gemini 1.5 Flash | Generate grounded answers from retrieved context |
| RAG Chain | LangChain `RetrievalQA` | Orchestrate retrieval + generation |

### Key benefits of RAG over plain LLM:
- ✅ Answers are grounded in YOUR dataset and knowledge base
- ✅ No hallucinations from the LLM's pre-training
- ✅ Can answer questions about your specific features and their correlations
- ✅ Explainable patient assessments combining ML + NLP

### Next steps to explore:
- Add more documents to the knowledge base (medical guidelines, papers)
- Build a web UI with Streamlit
- Add conversational memory so the AI remembers the conversation history
- Fine-tune the number of retrieved chunks (k) for better accuracy